# Hangman wordlist explorer

Play with the curated master wordlist and the `wordlist.py` loader.
Run from anywhere in the project (repo root, `math/`, or `math/notebooks/`)
— the setup cell finds `wordlist.py` and imports it.

The list is `data/wordlist_master.txt` (`word<TAB>zipf`, frequency-sorted),
built by `math/build_wordlist.py`. `zipf` is wordfreq's frequency scale:
higher = more common (4.0 ≈ 10/million, 3.0 ≈ 1/million, 2.0 = the file floor).
The loader's default cutoff is `DEFAULT_ZIPF = 2.5`.

In [ ]:
from pathlib import Path
import sys

def find_math_dir():
    """Find the directory containing wordlist.py (the math/ dir)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (candidate / 'wordlist.py').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find wordlist.py. Open this notebook from the project root, '
        'math/, or math/notebooks/.'
    )

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import wordlist
from wordlist import load, tier, Word, DEFAULT_ZIPF, TIERS

print(f'imported wordlist from {math_dir}')
print(f'DEFAULT_ZIPF = {DEFAULT_ZIPF}')

## The whole list at a glance

In [ ]:
full = wordlist._raw()   # every stored word, frequency order, no cutoff
print(f'{len(full):,} words total, zipf {full[-1].zipf:.2f} .. {full[0].zipf:.2f}')
print(f'{len(load()):,} words at the default cutoff (zipf >= {DEFAULT_ZIPF})')
print()
print('most common 15:', ', '.join(w.text for w in full[:15]))
print('rarest 15:     ', ', '.join(w.text for w in full[-15:]))

## Tune the cutoff

Change `CUTOFF` and re-run to see how many words survive and what the
boundary looks like. Words *just above* the cutoff are the hardest ones
still included — a good gut-check for whether the cutoff feels right.

In [ ]:
CUTOFF = 2.5   # <-- try 3.0, 2.8, 2.2, 2.0 ...

words = load(min_zipf=CUTOFF)
print(f'zipf >= {CUTOFF}: {len(words):,} words\n')

# 30 words sitting right at the cutoff (the marginal, hardest-kept words)
near = [w for w in words if w.zipf < CUTOFF + 0.06]
print(f'{len(near)} words within 0.05 of the cutoff — sample:')
print(', '.join(w.text for w in near[:30]))

## Filter by length and frequency

`load()` takes `min_len`, `max_len`, `min_zipf`, `max_zipf`. Handy combo:
a `min_zipf`/`max_zipf` *band* isolates "known but not trivially common"
words — good for a harder round.

In [ ]:
# e.g. 6-letter words in the 'known but not everyday' band
sel = load(min_len=6, max_len=6, min_zipf=2.5, max_zipf=3.5)
print(f'{len(sel):,} six-letter words, zipf 2.5-3.5')
print(', '.join(w.text for w in sel[:40]))

## Difficulty tiers

`TIERS` are presets (zipf band + length range); `tier(name)` returns the pool.
Edit `TIERS` in `wordlist.py` to retune them.

In [ ]:
for name, spec in TIERS.items():
    pool = tier(name)
    print(f'{name:<7} {len(pool):>6,} words   {spec}')
    print('        e.g.', ', '.join(w.text for w in pool[:10]))

## Pick a word (the gameplay use)

For an actual game you want a *random* word from a pool, not the most common
one. `pick()` samples uniformly — pass a tier name or explicit filters.

In [ ]:
import random

def pick(difficulty=None, *, n=1, seed=None, **filters):
    """Random word(s). Give a tier name (difficulty=) OR load() filters."""
    pool = tier(difficulty) if difficulty else load(**filters)
    rng = random.Random(seed)
    picks = rng.sample(pool, min(n, len(pool)))
    return picks[0] if n == 1 else picks

print('easy: ', pick('easy').text)
print('medium:', pick('medium').text)
print('hard: ', pick('hard').text)
print()
print('10 random medium words:')
print(', '.join(w.text for w in pick('medium', n=10)))

## Distributions

Text histograms (no plotting deps). How word length and frequency are spread
across the list at the current cutoff — useful for sanity-checking difficulty.

In [ ]:
from collections import Counter

def bar(count, scale):
    return '█' * int(count / scale)

words = load(min_zipf=CUTOFF)

print(f'Word length distribution (zipf >= {CUTOFF}):')
by_len = Counter(len(w.text) for w in words)
scale = max(by_len.values()) / 40
for length in range(min(by_len), max(by_len) + 1):
    c = by_len.get(length, 0)
    print(f'  {length:>2} | {bar(c, scale):<40} {c:,}')

print(f'\nWords per 0.5-zipf band:')
by_band = Counter((int(w.zipf * 2) / 2) for w in words)
scale = max(by_band.values()) / 40
for band in sorted(by_band, reverse=True):
    c = by_band[band]
    print(f'  {band:>4.1f} | {bar(c, scale):<40} {c:,}')

## Look up a specific word

Check whether a word made the list and at what frequency — handy when you
wonder "why isn't X in here?" (usually: filtered as a proper noun, or below
the floor).

In [ ]:
index = {w.text: w for w in wordlist._raw()}

for q in ['running', 'apricot', 'apricots', 'xylophone', 'the', 'amtrak']:
    w = index.get(q)
    print(f'{q:<12}', f'zipf {w.zipf:.2f}, rank {w.rank:,}' if w else 'not in list')